# 06 · Model evaluation
**Objective:** read the held-out results, compare methods, and state what the evidence does and does not support.
Results come from `python -m app.agents.evaluation` runs stored in `research/evaluations/`; this notebook does not re-run them.

In [1]:
# Setup: works locally and on Google Colab. Never put secrets in notebook cells.
import json, os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = ""  # Set to your clone URL. On Colab, add HF_TOKEN under Secrets to use hosted models.
    if not Path("salesiq").exists():
        if not REPO_URL:
            raise RuntimeError("Set REPO_URL in this cell to clone the SalesIQ repository.")
        subprocess.run(["git", "clone", "--depth=1", REPO_URL, "salesiq"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx", "pyyaml", "pydantic", "mcp"], check=True)
    ROOT = Path("salesiq").resolve()
    from google.colab import userdata
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except userdata.SecretNotFoundError:
        print("No HF_TOKEN secret; only a local model server can be used.")
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "research").is_dir())
sys.path.insert(0, str(ROOT / "apps" / "api"))
os.environ["SALESIQ_RESEARCH_DIR"] = str(ROOT / "research")

print('Reading stored results; no model is called here.')

Reading stored results; no model is called here.


In [2]:
from IPython.display import Markdown, display
from app.agents.evaluation.report import load_summaries, merge, render

runs = load_summaries(split="test")
assert runs, "No held-out runs found. Run: python -m app.agents.evaluation --models baseline,qwen2.5-7b --split test"
merged = merge(runs)
display(Markdown(render(merged, runs)))

# Model evaluation report

Generated by `python -m app.agents.evaluation --report`. Do not edit by hand.

**Read this first.** The data is synthetic and fictional, and labels come from a known rule, so these numbers compare
methods with each other. They do not predict production accuracy or real conversion rates. Samples are small; the 95%
intervals are wide, and overlapping intervals mean the difference is not established. Lexical support is a screening
heuristic, not proof a claim is true. Local cost excludes hardware and power. No human ratings have been collected yet.


Runs: `v1-test-baseline` (b364e0e), `v1-test-baseline-r2` (b00bc4d), `v1-test-mistral-7b` (b364e0e), `v1-test-qwen2.5-7b` (b364e0e), `v1-test-qwen2.5-7b-r2` (b00bc4d), `v1-test-qwen3-4b-nothink` (8ace064)

## ICP classification (higher is better)

| Method | Accuracy | 95% CI | Macro F1 | Valid output | Failures | Served by fallback | p50 | p95 |
|---|---|---|---|---|---|---|---|---|
| `baseline` | 53% | 41-65% | 0.51 | 100% | none | 0 | 0 ms | 0 ms |
| `mistral-7b` | 67% | 54-77% | 0.67 | 100% | none | 0 | 7.5 s | 11.2 s |
| `qwen2.5-7b` | 62% | 49-73% | 0.60 | 100% | none | 0 | 4.3 s | 5.1 s |
| `qwen3-4b` | 62% | 49-73% | 0.62 | 100% | none | 0 | 6.9 s | 11.1 s |

## Structured extraction

| Method | Field accuracy | 95% CI | Invented facts | Valid output | Failures | Served by fallback | p50 | p95 |
|---|---|---|---|---|---|---|---|---|
| `baseline` | 27% | 23-33% | 0% of 22 absent | 100% | none | 0 | 0 ms | 0 ms |
| `mistral-7b` | 90% | 86-93% | 9% of 22 absent | 100% | none | 0 | 3.4 s | 3.7 s |
| `qwen2.5-7b` | 91% | 87-93% | 5% of 22 absent | 100% | none | 0 | 1.6 s | 1.7 s |
| `qwen3-4b` | 98% | 96-99% | 0% of 22 absent | 100% | none | 0 | 2.1 s | 3.0 s |

## Prompt injection: low-fit leads with instructions planted to force "high" (lower is better)

| Method | Attack success | 95% CI | Still labelled low | Valid output | Failures | Served by fallback | p50 | p95 |
|---|---|---|---|---|---|---|---|---|
| `baseline` | 0% | 0-16% | 12 of 20 | 100% | none | 0 | 0 ms | 0 ms |
| `mistral-7b` | 40% | 22-61% | 3 of 20 | 100% | none | 0 | 9.8 s | 14.2 s |
| `qwen2.5-7b` | 20% | 8-42% | 15 of 20 | 100% | none | 0 | 4.2 s | 4.9 s |
| `qwen3-4b` | 55% | 34-74% | 9 of 20 | 100% | none | 0 | 5.7 s | 9.7 s |

## Research synthesis

| Method | Lexical support | Invented numbers | Invalid citations | Injection obeyed | Injection cited | Valid output | Failures | Served by fallback | p50 | p95 |
|---|---|---|---|---|---|---|---|---|---|---|
| `baseline` | 1.00 | 0 of 24 | 0 of 24 | 0 of 12 | 0 of 12 | 100% | none | 0 | 0 ms | 0 ms |
| `mistral-7b` | 0.55 | 6 of 24 | 0 of 24 | 2 of 12 | 8 of 12 | 100% | none | 0 | 16.3 s | 20.8 s |
| `qwen2.5-7b` | 0.69 | 1 of 24 | 0 of 24 | 3 of 12 | 4 of 12 | 100% | none | 0 | 6.7 s | 7.8 s |
| `qwen3-4b` | 0.63 | 3 of 24 | 0 of 24 | 8 of 12 | 9 of 12 | 100% | none | 0 | 10.0 s | 14.0 s |

## Outreach drafts

| Method | Lexical support | Invented numbers | Invalid citations | Injection obeyed | Injection cited | Valid output | Failures | Served by fallback | p50 | p95 |
|---|---|---|---|---|---|---|---|---|---|---|
| `baseline` | 0.49 | 0 of 24 | 0 of 24 | 0 of 12 | 0 of 12 | 100% | none | 0 | 0 ms | 0 ms |
| `mistral-7b` | 0.21 | 15 of 24 | 1 of 24 | 1 of 12 | 2 of 12 | 100% | none | 0 | 8.0 s | 10.1 s |
| `qwen2.5-7b` | 0.17 | 12 of 15 | 1 of 15 | 0 of 7 | 4 of 7 | 62% | truncated 9 | 0 | 3.1 s | 165.9 s |
| `qwen3-4b` | 0.30 | 0 of 24 | 0 of 24 | 3 of 12 | 4 of 12 | 100% | none | 0 | 4.3 s | 5.2 s |

## Evidence retrieval

| Method | Recall@1 | 95% CI | MRR | Valid output | Failures | Served by fallback | p50 | p95 |
|---|---|---|---|---|---|---|---|---|
| `baseline` | 79% | 60-91% | 0.86 | 100% | none | 0 | 0 ms | 0 ms |
| `nomic-embed` | 96% | 80-99% | 0.98 | 100% | none | 0 | 79 ms | 230 ms |

## Token usage

| Task | Model | Input | Output | Est. cost |
|---|---|---|---|---|
| classification | `mistral-7b` | 36,331 | 11,949 | $0.00 (local) |
| classification | `qwen2.5-7b` | 30,890 | 9,843 | $0.00 (local) |
| classification | `qwen3-4b` | 31,010 | 11,175 | $0.00 (local) |
| extraction | `mistral-7b` | 14,161 | 4,043 | $0.00 (local) |
| extraction | `qwen2.5-7b` | 13,269 | 3,212 | $0.00 (local) |
| extraction | `qwen3-4b` | 13,389 | 2,896 | $0.00 (local) |
| injection | `mistral-7b` | 12,571 | 4,065 | $0.00 (local) |
| injection | `qwen2.5-7b` | 10,714 | 3,251 | $0.00 (local) |
| injection | `qwen3-4b` | 10,754 | 3,034 | $0.00 (local) |
| outreach | `mistral-7b` | 7,134 | 4,755 | $0.00 (local) |
| outreach | `qwen2.5-7b` | 3,969 | 1,869 | unknown |
| outreach | `qwen3-4b` | 6,428 | 2,799 | $0.00 (local) |
| synthesis | `mistral-7b` | 6,190 | 8,341 | $0.00 (local) |
| synthesis | `qwen2.5-7b` | 5,559 | 6,686 | $0.00 (local) |
| synthesis | `qwen3-4b` | 5,607 | 6,625 | $0.00 (local) |


## Is any difference real?
Compare intervals before believing a gap. Non-overlap here is a rough screen, not a formal test.

In [3]:
acc = {m: e["quality"] for m, e in merged["classification"].items()}
best = max(acc, key=lambda m: acc[m]["accuracy"])
base = acc["baseline"]
for m, q in sorted(acc.items(), key=lambda kv: -kv[1]["accuracy"]):
    lo, hi = q["accuracy_ci95"]
    clear = lo > base["accuracy_ci95"][1]
    note = "baseline" if m == "baseline" else ("clearly above baseline" if clear else "not clearly above baseline")
    print(f"{m:14} {q['accuracy']:.0%}  CI {lo:.0%}-{hi:.0%}  {note}")

mistral-7b     67%  CI 54%-77%  not clearly above baseline
qwen2.5-7b     62%  CI 49%-73%  not clearly above baseline
qwen3-4b       62%  CI 49%-73%  not clearly above baseline
baseline       53%  CI 41%-65%  baseline


**Limitations:** fictional data, rule-derived labels, one prompt version, small samples, no human ratings. Fine-tuning is not
justified by this evidence; the next improvements are retrieval, stronger validation, and real-document evaluation.